In [ ]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Path to DEM (notebook is in output/, data is one level up)
dem_path = Path.cwd().parent / "data" / "ldem_85s_20m_float.lbl"

# Open DEM
dem = rasterio.open(dem_path)

# Read first band
elevation = dem.read(1)

# Print information
print("=" * 50)
print("DEM Information")
print("=" * 50)

print(f"Width        : {dem.width}")
print(f"Height       : {dem.height}")
print(f"Bands        : {dem.count}")
print(f"Data Type    : {dem.dtypes[0]}")
print(f"CRS          : {dem.crs}")
print(f"Transform    :")
print(dem.transform)

print("\nElevation Statistics")
print("---------------------")
print(f"Minimum : {np.nanmin(elevation)}")
print(f"Maximum : {np.nanmax(elevation)}")
print(f"Mean    : {np.nanmean(elevation)}")

# Display DEM

plt.figure(figsize=(10, 8))
plt.imshow(elevation, cmap="gray")
plt.colorbar(label="Elevation")
plt.title("LOLA DEM")
plt.show()

In [ ]:
import rasterio
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from rasterio.features import rasterize

base = Path.cwd().parent  # ISRO_Hackathon/

# ============================================================
# STEP 2 : Load DEM
# ============================================================

dem = rasterio.open(base / "data" / "ldem_85s_20m_float.lbl")

print("✓ DEM Loaded")
print("  DEM CRS:", dem.crs)

# ============================================================
# Load PSR Shapefile
# ============================================================

psr = gpd.read_file(base / "data" / "LPSR_80S_20MPP_ADJ.shp")

print("✓ PSR Loaded")
print("  PSR CRS:", psr.crs)

# ============================================================
# Show Information
# ============================================================

print("\nNumber of polygons :", len(psr))
print("\nColumns")
print(psr.columns)

# ============================================================
# Match Coordinate Systems
# ============================================================

if psr.crs is None:
    print("\nPSR has no CRS — assigning DEM CRS (same data source)")
    psr = psr.set_crs(dem.crs)
elif psr.crs != dem.crs:
    print("\nReprojecting PSR...")
    psr = psr.to_crs(dem.crs)
else:
    print("\nCRS already matches")

# ============================================================
# Rasterize
# ============================================================

mask = rasterize(
    [(geom, 1) for geom in psr.geometry],
    out_shape=(dem.height, dem.width),
    transform=dem.transform,
    fill=0,
    dtype="uint8"
)

print("\n✓ Rasterization Finished")

# ============================================================
# Statistics
# ============================================================

total_pixels = mask.size
psr_pixels = np.sum(mask)

print("\nTotal Pixels :", total_pixels)
print("PSR Pixels   :", psr_pixels)
print("Coverage (%) :", (psr_pixels / total_pixels) * 100)

# ============================================================
# Display
# ============================================================

plt.figure(figsize=(10, 10))
plt.imshow(mask, cmap="gray")
plt.title("PSR Mask")
plt.xlabel("Column")
plt.ylabel("Row")
plt.show()

# ============================================================
# Save Raster
# ============================================================

out_path = Path.cwd() / "PSR_mask.tif"  # saves into output/

with rasterio.open(
    out_path, "w",
    driver="GTiff",
    height=dem.height,
    width=dem.width,
    count=1,
    dtype=mask.dtype,
    crs=dem.crs,
    transform=dem.transform
) as dst:
    dst.write(mask, 1)

print("\n✓ Saved", out_path)

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

base = Path.cwd().parent  # ISRO_Hackathon/

# ============================================================
# STEP 3 : Load DEM
# ============================================================

dem = rasterio.open(base / "data" / "ldem_85s_20m_float.lbl")

elevation = dem.read(1)

# Convert km to meters (LOLA DEM stores elevation in km)
elevation = elevation * 1000

print("✓ DEM Loaded")

# ============================================================
# Pixel Resolution
# ============================================================

cellsize = dem.transform.a  # 20 m

print("Cell Size :", cellsize)

# ============================================================
# Calculate X and Y Gradients
# ============================================================

dz_dy, dz_dx = np.gradient(elevation, cellsize)

print("✓ Gradient Computed")

# ============================================================
# Calculate Slope
# ============================================================

slope = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
slope_deg = np.degrees(slope)

print("✓ Slope Computed")

# ============================================================
# Calculate Aspect
# ============================================================

aspect = np.degrees(np.arctan2(-dz_dx, dz_dy))
aspect = (aspect + 360) % 360

print("✓ Aspect Computed")

# ============================================================
# Statistics
# ============================================================

print("\nSlope Statistics")
print("----------------")
print("Minimum :", np.nanmin(slope_deg))
print("Maximum :", np.nanmax(slope_deg))
print("Mean    :", np.nanmean(slope_deg))

# ============================================================
# Display Slope
# ============================================================

plt.figure(figsize=(10, 8))
plt.imshow(slope_deg, cmap="terrain")
plt.colorbar(label="Slope (degrees)")
plt.title("Slope Map")
plt.tight_layout()
plt.show()

# ============================================================
# Display Aspect
# ============================================================

plt.figure(figsize=(10, 8))
plt.imshow(aspect, cmap="hsv")
plt.colorbar(label="Aspect (degrees)")
plt.title("Aspect Map")
plt.tight_layout()
plt.show()

# ============================================================
# Save Slope Raster
# ============================================================

with rasterio.open(
    Path.cwd() / "slope.tif", "w",
    driver="GTiff",
    height=dem.height,
    width=dem.width,
    count=1,
    dtype=slope_deg.dtype,
    crs=dem.crs,
    transform=dem.transform
) as dst:
    dst.write(slope_deg, 1)

print("✓ Saved slope.tif")

# ============================================================
# Save Aspect Raster
# ============================================================

with rasterio.open(
    Path.cwd() / "aspect.tif", "w",
    driver="GTiff",
    height=dem.height,
    width=dem.width,
    count=1,
    dtype=aspect.dtype,
    crs=dem.crs,
    transform=dem.transform
) as dst:
    dst.write(aspect, 1)

print("✓ Saved aspect.tif")

In [ ]:
# Hillshade — kept for visualisation only.
# NOT used as illumination input (shadow casting below replaces it).
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

base = Path.cwd().parent  # ISRO_Hackathon/

# ============================================================
# STEP 4 : Load DEM
# ============================================================

dem = rasterio.open(base / "data" / "ldem_85s_20m_float.lbl")

elevation = dem.read(1)

# Convert km to meters
elevation = elevation * 1000

print("✓ DEM Loaded")

# ============================================================
# Pixel Size
# ============================================================

cellsize = dem.transform.a

# ============================================================
# Calculate Gradient
# ============================================================

dz_dy, dz_dx = np.gradient(elevation, cellsize)

# ============================================================
# Calculate Slope
# ============================================================

slope = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))

# ============================================================
# Calculate Aspect
# ============================================================

aspect = np.arctan2(-dz_dx, dz_dy)

# ============================================================
# Sun Position
# ============================================================

sun_azimuth = 180    # degrees
sun_elevation = 2    # degrees

azimuth_rad = np.radians(sun_azimuth)
elevation_rad = np.radians(sun_elevation)

print("Sun Azimuth :", sun_azimuth)
print("Sun Elevation :", sun_elevation)

# ============================================================
# Hillshade Equation
# ============================================================

hillshade = (
    np.sin(elevation_rad) * np.cos(slope)
    + np.cos(elevation_rad) * np.sin(slope) * np.cos(azimuth_rad - aspect)
)

# ============================================================
# Normalize
# ============================================================

hillshade = np.clip(hillshade, 0, 1)

print("✓ Hillshade Computed")

# ============================================================
# Statistics
# ============================================================

print()
print("Minimum :", hillshade.min())
print("Maximum :", hillshade.max())
print("Mean    :", hillshade.mean())

# ============================================================
# Display
# ============================================================

plt.figure(figsize=(10, 10))
plt.imshow(hillshade, cmap="gray")
plt.title("Hillshade")
plt.colorbar(label="Illumination")
plt.tight_layout()
plt.show()

# ============================================================
# Save Raster
# ============================================================

with rasterio.open(
    Path.cwd() / "hillshade.tif", "w",
    driver="GTiff",
    height=dem.height,
    width=dem.width,
    count=1,
    dtype="float32",
    crs=dem.crs,
    transform=dem.transform
) as dst:
    dst.write(hillshade.astype("float32"), 1)

print("✓ Saved hillshade.tif")

In [ ]:
import math
import numpy as np
from pathlib import Path
import rasterio
from numba import njit, prange

BASE = Path.cwd().parent
OUT_DIR = BASE / 'results'
CELLSIZE      = 20.0
SUN_AZIMUTH   = 0.0    # deg, 0=North 90=East
SUN_ELEVATION = 1.54   # deg — peak solar elevation at ~89.5°S
MAX_DISTANCE  = 2500   # 2500 px × 20 m = 50 km

import math
from numba import njit, prange

# Solar shadow map — replaces hillshade > threshold
# Pixel P is illuminated iff no terrain toward the sun exceeds sun elevation:
#   (h_d - h_P) / dist_d  <  tan(SUN_ELEVATION)   for all d along sun ray
# Same derivation as the DPSR LOS kernel; equivalent to GRASS r.sunmask.

_sun_tan  = math.tan(math.radians(SUN_ELEVATION))
_az_rad   = math.radians(SUN_AZIMUTH)
_steps    = np.arange(1, MAX_DISTANCE + 1, dtype=np.float64)
_sun_dr   = np.round(-math.cos(_az_rad) * _steps).astype(np.int32)
_sun_dc   = np.round( math.sin(_az_rad) * _steps).astype(np.int32)
_sun_dist = (np.sqrt(_sun_dr.astype(np.float64)**2 +
                     _sun_dc.astype(np.float64)**2) * CELLSIZE).astype(np.float32)
_sun_dist = np.where(_sun_dist < 1.0, np.float32(1.0), _sun_dist)

@njit(parallel=True, cache=True, fastmath=True)
def _shadow_kernel(elevation, sun_dr, sun_dc, sun_dist, sun_tan):
    H, W = elevation.shape
    n    = sun_dr.shape[0]
    out  = np.ones((H, W), dtype=np.uint8)   # default: illuminated
    for row in prange(H):
        for col in range(W):
            cur_h = elevation[row, col]
            for d in range(n):
                r = row + sun_dr[d]
                c = col + sun_dc[d]
                if r < 0 or r >= H or c < 0 or c >= W:
                    break
                if (elevation[r, c] - cur_h) / sun_dist[d] >= sun_tan:
                    out[row, col] = 0
                    break
    return out

print(f'Solar shadow map  az={SUN_AZIMUTH:.0f} el={SUN_ELEVATION:.2f} deg ')
import time as _t; _t0 = _t.time()
illumination = _shadow_kernel(elevation, _sun_dr, _sun_dc, _sun_dist, _sun_tan)
print(f'  done {_t.time()-_t0:.1f} s   lit={illumination.mean()*100:.1f}%')

# Save
meta = {k: v for k,v in rasterio.open(BASE/'data'/'ldem_85s_20m_float.lbl').meta.items()}
meta.update(dtype='uint8', count=1, compress='lzw')
with rasterio.open(OUT_DIR / 'illumination.tif', 'w', **meta) as dst:
    dst.write(illumination, 1)
print(f'Saved illumination.tif   illuminated={illumination.mean()*100:.1f}%')


In [ ]:
import rasterio
import matplotlib.pyplot as plt
from pathlib import Path

base = Path.cwd().parent  # ISRO_Hackathon/

dem_arr   = rasterio.open(base / "data" / "ldem_85s_20m_float.lbl").read(1)
illum_arr = rasterio.open(Path.cwd() / "illumination.tif").read(1)

plt.figure(figsize=(10, 10))
plt.imshow(dem_arr, cmap="gray")
plt.imshow(illum_arr, cmap="Reds", alpha=0.35)
plt.title("DEM + Illumination Overlay")
plt.show()


In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# =====================================================
# Load PSR Mask
# =====================================================

psr_ds   = rasterio.open(Path.cwd() / "PSR_mask.tif")
psr_mask = psr_ds.read(1)

print("✓ PSR Mask Loaded")

# =====================================================
# Find PSR Pixels
# =====================================================

rows, cols = np.where(psr_mask == 1)

print()
print("Total PSR Pixels :", len(rows))

# =====================================================
# Create Coordinate List
# =====================================================

psr_pixels = list(zip(rows.tolist(), cols.tolist()))

print()
print("First 10 PSR Pixels")
for p in psr_pixels[:10]:
    print(p)

# =====================================================
# Display
# =====================================================

plt.figure(figsize=(8, 8))
plt.imshow(psr_mask, cmap="gray")
plt.scatter(cols[:3000], rows[:3000], s=1, color="red")
plt.title("Detected PSR Pixels")
plt.show()

# =====================================================
# Save Coordinates
# =====================================================

np.save(Path.cwd() / "psr_pixels.npy", np.array(psr_pixels))

print()
print("✓ Saved psr_pixels.npy")


In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

base = Path.cwd().parent  # ISRO_Hackathon/

# ======================================================
# Load DEM
# ======================================================

dem       = rasterio.open(base / "data" / "ldem_85s_20m_float.lbl")
elevation = dem.read(1) * 1000  # km → m

# ======================================================
# Load Illumination Map
# ======================================================

illumination = rasterio.open(Path.cwd() / "illumination.tif").read(1)

# ======================================================
# Load PSR Coordinates
# ======================================================

psr_pixels = np.load(Path.cwd() / "psr_pixels.npy")

# ======================================================
# Pick ONE PSR Pixel
# ======================================================

row, col = int(psr_pixels[0, 0]), int(psr_pixels[0, 1])

print("Testing Pixel:", row, col)

# ======================================================
# One Ray Direction (45°)
# ======================================================

angle        = np.radians(45)
max_distance = 2500
cellsize     = 20

# ======================================================
# Walk Along Ray (for plotting)
# ======================================================

ray_rows, ray_cols = [], []

for d in range(1, max_distance):
    r = int(row + d * np.sin(angle))
    c = int(col + d * np.cos(angle))
    if r < 0 or r >= elevation.shape[0] or c < 0 or c >= elevation.shape[1]:
        break
    ray_rows.append(r)
    ray_cols.append(c)

# ======================================================
# Plot Ray
# ======================================================

plt.figure(figsize=(10, 10))
plt.imshow(elevation, cmap="gray")
plt.scatter(col, row, color="red", s=60, label="PSR Pixel")
plt.plot(ray_cols, ray_rows, color="yellow", linewidth=2, label="Ray")
plt.legend()
plt.title("Single Ray")
plt.show()

# ======================================================
# Visibility Check (illumination check BEFORE horizon update)
# ======================================================

current_height = elevation[row, col]
highest_angle  = -999.0
visible        = False

for d in range(1, max_distance):
    r = int(row + d * np.sin(angle))
    c = int(col + d * np.cos(angle))
    if r < 0 or r >= elevation.shape[0] or c < 0 or c >= elevation.shape[1]:
        break

    target_height  = elevation[r, c]
    distance       = d * cellsize
    terrain_angle  = np.degrees(np.arctan((target_height - current_height) / distance))

    # Check visibility before updating horizon
    if illumination[r, c] == 1 and terrain_angle >= highest_angle:
        visible = True
        break

    if terrain_angle > highest_angle:
        highest_angle = terrain_angle

if visible:
    print("Visible illuminated terrain")
else:
    print("Blocked — Candidate DPSR")


In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

base = Path.cwd().parent  # ISRO_Hackathon/

# ============================================================
# Load DEM
# ============================================================

dem       = rasterio.open(base / "data" / "ldem_85s_20m_float.lbl")
elevation = dem.read(1) * 1000

# ============================================================
# Load Illumination Map
# ============================================================

illumination = rasterio.open(Path.cwd() / "illumination.tif").read(1)

# ============================================================
# Load PSR Pixels
# ============================================================

psr_pixels = np.load(Path.cwd() / "psr_pixels.npy")

# ============================================================
# Select ONE PSR Pixel
# ============================================================

row, col = int(psr_pixels[0, 0]), int(psr_pixels[0, 1])

print("Testing Pixel:", row, col)

# ============================================================
# Parameters
# ============================================================

cellsize       = 20
max_distance   = 2500
angles         = np.arange(0, 360, 5)
current_height = elevation[row, col]
visible        = False

# ============================================================
# Plot Setup
# ============================================================

plt.figure(figsize=(10, 10))
plt.imshow(elevation, cmap="gray")
plt.scatter(col, row, color="red", s=60)

# ============================================================
# Loop Over Every Direction
# ============================================================

for angle_deg in angles:
    angle         = np.radians(angle_deg)
    highest_angle = -999.0
    ray_x, ray_y  = [], []

    for d in range(1, max_distance):
        r = int(row + d * np.sin(angle))
        c = int(col + d * np.cos(angle))
        if r < 0 or r >= elevation.shape[0] or c < 0 or c >= elevation.shape[1]:
            break

        ray_y.append(r)
        ray_x.append(c)

        target_height = elevation[r, c]
        distance      = d * cellsize
        terrain_angle = np.degrees(np.arctan((target_height - current_height) / distance))

        # Check visibility before updating horizon
        if illumination[r, c] == 1 and terrain_angle >= highest_angle:
            visible = True
            break

        if terrain_angle > highest_angle:
            highest_angle = terrain_angle

    plt.plot(ray_x, ray_y, linewidth=0.5)

    if visible:
        break

# ============================================================
# Result
# ============================================================

if visible:
    print()
    print("Visible illuminated terrain — NOT DPSR")
else:
    print()
    print("No illuminated terrain visible — Candidate DPSR")

plt.title("360° Horizon Analysis")
plt.show()


In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

base = Path.cwd().parent  # ISRO_Hackathon/

# ============================================================
# Load DEM
# ============================================================

dem       = rasterio.open(base / "data" / "ldem_85s_20m_float.lbl")
elevation = dem.read(1) * 1000

# ============================================================
# Load Illumination
# ============================================================

illumination = rasterio.open(Path.cwd() / "illumination.tif").read(1)

# ============================================================
# Load PSR Mask
# ============================================================

psr_mask = rasterio.open(Path.cwd() / "PSR_mask.tif").read(1)

# ============================================================
# Extract PSR Pixels
# ============================================================

rows, cols = np.where(psr_mask == 1)

print("Total PSR Pixels :", len(rows))

# ============================================================
# Create Output Raster
# ============================================================

dpsr = np.zeros_like(psr_mask, dtype=np.uint8)

# ============================================================
# Parameters
# ============================================================

cellsize     = 20
max_distance = 2500
angles       = np.arange(0, 360, 5)

# ============================================================
# DPSR Function
# ============================================================

def is_dpsr(row, col):
    current_height = elevation[row, col]

    for angle_deg in angles:
        angle         = np.radians(angle_deg)
        highest_angle = -999.0

        for d in range(1, max_distance):
            r = int(row + d * np.sin(angle))
            c = int(col + d * np.cos(angle))
            if r < 0 or r >= elevation.shape[0] or c < 0 or c >= elevation.shape[1]:
                break

            target_height = elevation[r, c]
            distance      = d * cellsize
            terrain_angle = np.degrees(np.arctan((target_height - current_height) / distance))

            # Check visibility before updating horizon
            if illumination[r, c] == 1 and terrain_angle >= highest_angle:
                return False  # can see illuminated terrain → not DPSR

            if terrain_angle > highest_angle:
                highest_angle = terrain_angle

    return True  # no illuminated terrain visible from any direction → DPSR

# ============================================================
# Process All PSR Pixels
# ============================================================

count = 0

for row, col in zip(rows, cols):
    if is_dpsr(int(row), int(col)):
        dpsr[row, col] = 1
    count += 1
    if count % 1000 == 0:
        print(f"{count} / {len(rows)} pixels processed")

# ============================================================
# Display
# ============================================================

plt.figure(figsize=(10, 10))
plt.imshow(dpsr, cmap="gray")
plt.title("Generated DPSR")
plt.show()

# ============================================================
# Save
# ============================================================

with rasterio.open(
    Path.cwd() / "DPSR.tif", "w",
    driver="GTiff",
    height=dem.height,
    width=dem.width,
    count=1,
    dtype="uint8",
    crs=dem.crs,
    transform=dem.transform,
) as dst:
    dst.write(dpsr, 1)

print()
print("===================================")
print("DPSR generation completed")
print("Saved : DPSR.tif")
print("===================================")


In [ ]:
# Run the optimised DPSR extraction
# Prerequisites: cells 0-6 must have run first (PSR_mask.tif, illumination.tif)
import subprocess, sys
subprocess.run([sys.executable, r'C:\Users\navne\Desktop\ISRO_Hackathon\dpsr_fast.py'], check=True)

In [ ]:
# ── Spot-check validation ─────────────────────────────────────────────────────
# Update row/col after inspecting the DEM in QGIS.
# A rim pixel should see illuminated terrain (NOT DPSR).
# A deep interior pixel should not (IS DPSR).
print("Spot-check validation:")
checks = [
    ("Shackleton rim (expect NOT DPSR)",  7580, 7580, False),
    ("Shackleton interior (expect DPSR)", 7584, 7584, True),
]
for label, row, col, expect in checks:
    d_val = bool(dpsr_raster[row, col])
    ok = (d_val == expect)
    print(f"  [{'OK' if ok else 'CHECK'}] {label}")
    print(f"         elev={elevation[row,col]:.0f}m  "
          f"illum={illumination[row,col]}  "
          f"psr={psr_mask[row,col]}  dpsr={int(d_val)}  "
          f"(expected dpsr={int(expect)})")
